# AI Stock Agent — Demo Notebook

This notebook demonstrates the AI Stock Agent deployed on AWS Bedrock AgentCore.
It authenticates via Cognito (demonstrating the user auth flow), then invokes the
agent through the AgentCore SDK (`invoke_agent_runtime`) which returns SSE-streamed
responses for each of the 5 required queries.

**Prerequisites:**
- Infrastructure deployed via `terraform apply`
- Cognito test user created (see README)
- AWS credentials configured (for boto3 SDK calls)
- `pip install boto3`

## Cell 1: Configuration

Set the deployment parameters from Terraform outputs.

In [1]:
import os
from pathlib import Path

from dotenv import dotenv_values, load_dotenv

# Load from .env.notebook at the project root (works in Cursor and Jupyter)
env_path = (
    Path(__file__).resolve().parent.parent / ".env.notebook"
    if "__file__" in dir()
    else Path("../.env.notebook")
)
load_dotenv(env_path, override=False)

# Prepend PATH from .env.notebook so credential_process tools (e.g. granted) are found
extra_path = dotenv_values(env_path).get("PATH", "")
if extra_path:
    os.environ["PATH"] = extra_path + os.pathsep + os.environ.get("PATH", "")

# ──────────────────────────────────────────────
# Values loaded from .env.notebook or shell env vars
# ──────────────────────────────────────────────
RUNTIME_ENDPOINT_ARN = os.environ.get("RUNTIME_ENDPOINT_ARN", "")
COGNITO_USER_POOL_ID = os.environ.get("COGNITO_USER_POOL_ID", "")
COGNITO_CLIENT_ID = os.environ.get("COGNITO_CLIENT_ID", "")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

COGNITO_USERNAME = os.environ.get("COGNITO_USERNAME", "testuser@example.com")
COGNITO_PASSWORD = os.environ.get("COGNITO_PASSWORD", "")

assert RUNTIME_ENDPOINT_ARN, "RUNTIME_ENDPOINT_ARN not set — check .env.notebook"
assert COGNITO_USER_POOL_ID, "COGNITO_USER_POOL_ID not set — check .env.notebook"

# Split endpoint ARN into runtime ARN + qualifier (endpoint name)
# e.g. ".../runtime/ID/runtime-endpoint/ENDPOINT_NAME" → runtime ARN + qualifier
parts = RUNTIME_ENDPOINT_ARN.split("/runtime-endpoint/")
AGENT_RUNTIME_ARN = parts[0]  # arn:...:runtime/ID
ENDPOINT_QUALIFIER = parts[1] if len(parts) > 1 else None

print(f"Runtime ARN:  {AGENT_RUNTIME_ARN}")
print(f"Endpoint:     {ENDPOINT_QUALIFIER}")
print(f"User Pool ID: {COGNITO_USER_POOL_ID}")
print(f"Client ID:    {COGNITO_CLIENT_ID}")
print(f"Region:       {AWS_REGION}")

Runtime ARN:  arn:aws:bedrock-agentcore:us-east-1:070017892077:runtime/ai_stock_agent-uftT5aBUKd
Endpoint:     ai_stock_agent_endpoint
User Pool ID: us-east-1_P90t3dHd4
Client ID:    26i39i6ff0r9jbqtdviblhrrtq
Region:       us-east-1


## Cell 2: Helper Functions

- `authenticate()` — Obtain a JWT from Cognito via `InitiateAuth`
- `invoke_agent()` — Call `invoke_agent_runtime` via boto3 SDK with SSE stream parsing

In [2]:
import json
import uuid

import boto3


def authenticate(
    user_pool_id: str = COGNITO_USER_POOL_ID,
    client_id: str = COGNITO_CLIENT_ID,
    username: str = COGNITO_USERNAME,
    password: str = COGNITO_PASSWORD,
    region: str = AWS_REGION,
) -> dict:
    """Authenticate with Cognito and return the full auth result including tokens."""
    client = boto3.client("cognito-idp", region_name=region)
    response = client.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={
            "USERNAME": username,
            "PASSWORD": password,
        },
    )
    return response["AuthenticationResult"]


def invoke_agent(
    prompt: str,
    id_token: str,
    thread_id: "str | None" = None,
    stream: bool = True,
    runtime_arn: str = AGENT_RUNTIME_ARN,
    qualifier: "str | None" = ENDPOINT_QUALIFIER,
    region: str = AWS_REGION,
) -> str:
    """Invoke the agent via AgentCore SDK and print streaming tokens.

    Uses boto3 invoke_agent_runtime. The response is a StreamingBody
    that our container fills with SSE events (stream=True) or JSON.
    Returns the full concatenated response text.
    """
    thread_id = thread_id or str(uuid.uuid4())
    payload = json.dumps(
        {
            "prompt": prompt,
            "thread_id": thread_id,
            "stream": stream,
        }
    ).encode()

    client = boto3.client("bedrock-agentcore", region_name=region)
    invoke_kwargs = {
        "agentRuntimeArn": runtime_arn,
        "runtimeSessionId": thread_id,
        "contentType": "application/json",
        "accept": "text/event-stream" if stream else "application/json",
        "payload": payload,
    }
    if qualifier:
        invoke_kwargs["qualifier"] = qualifier

    response = client.invoke_agent_runtime(**invoke_kwargs)

    full_response = []
    body = response["response"]  # botocore StreamingBody

    raw = body.read()
    text = raw.decode("utf-8") if isinstance(raw, bytes) else raw

    # Try SSE parsing first (data: {...} lines)
    sse_found = False
    for line in text.splitlines():
        if line.startswith("data: "):
            sse_found = True
            data = json.loads(line[6:])
            if data.get("type") == "token":
                token = data["content"]
                print(token, end="", flush=True)
                full_response.append(token)
            elif data.get("type") == "end":
                break

    if not sse_found:
        # Not SSE — try JSON, then fall back to raw text
        try:
            result = json.loads(text)
            answer = result.get("response", str(result))
        except json.JSONDecodeError:
            answer = text
        print(answer)
        full_response.append(answer)

    print()
    return "".join(full_response)


print("Helpers loaded ✓")

Helpers loaded ✓


## Cell 3: Authenticate with Cognito

Obtain a JWT via `USER_PASSWORD_AUTH` flow.

In [3]:
auth_result = authenticate()
id_token = auth_result["IdToken"]

print(f"Access Token (first 40 chars): {auth_result['AccessToken'][:40]}...")
print(f"ID Token     (first 40 chars): {id_token[:40]}...")
print(f"Token Type: {auth_result['TokenType']}")
print(f"Expires In: {auth_result['ExpiresIn']}s")
print("\nAuthentication successful ✓")

Access Token (first 40 chars): eyJraWQiOiJzSkYrTHZFdEpcL1hseXFweVJPNlU0...
ID Token     (first 40 chars): eyJraWQiOiJ6MFZDNHpudWRSeStqZTU2Tks0a2Za...
Token Type: Bearer
Expires In: 3600s

Authentication successful ✓


## Cell 4: Query 1 — Real-Time Stock Price

> *"What is the stock price for Amazon right now?"*

Expected: Agent calls `retrieve_realtime_stock_price` with ticker AMZN.

In [4]:
response_1 = invoke_agent(
    "What is the stock price for Amazon right now?",
    id_token=id_token,
)

I'll get the current stock price for Amazon (AMZN) for you.Amazon's current stock price is **$220.79 USD**.

Here are some additional details:
- Previous close: $213.77
- Day's high: $226.00
- Day's low: $219.62
- Volume: 18,104,324 shares
- Last updated: April 8, 2026 at 15:17 UTC

The stock is up about $7.02 (+3.28%) from yesterday's close.


## Cell 5: Query 2 — Historical Stock Prices

> *"What were the stock prices for Amazon in Q4 last year?"*

Expected: Agent calls `retrieve_historical_stock_price` with Q4 2025 date range.

In [5]:
response_2 = invoke_agent(
    "What were the stock prices for Amazon in Q4 last year?",
    id_token=id_token,
)

I'll help you get Amazon's stock prices for Q4 of last year (2023). Let me retrieve the historical stock data for that period.Here's a summary of Amazon's stock prices during Q4 2023 (October-December):

## Q4 2023 Amazon Stock Price Summary

**Key Price Points:**
- **Q4 Opening Price (Oct 2):** $129.46
- **Q4 Closing Price (Dec 29):** $151.94
- **Quarterly Gain:** +17.4%

**Monthly Breakdown:**

**October 2023:**
- Range: $118.35 - $134.48
- Started at $129.46, ended at $133.09
- Notable volatility with a significant dip to ~$119 on Oct 26th, followed by strong recovery

**November 2023:**
- Range: $133.09 - $149.26
- Strong upward trend throughout the month
- Ended at $146.09, gaining momentum

**December 2023:**
- Range: $142.81 - $155.63
- Peak reached on Dec 20th at $155.63
- Finished the quarter at $151.94

**Notable Events:**
- **Significant Rally:** Strong performance in late October after earnings
- **Peak Performance:** Mid-to-late December saw the highest prices of the quart

## Cell 6: Query 3 — Cross-Reference Stock + Reports

> *"Compare Amazon's recent stock performance to what analysts predicted in their reports"*

Expected: Agent calls both yfinance (historical) and RAG (earnings reports).

In [6]:
response_3 = invoke_agent(
    "Compare Amazon's recent stock performance to what analysts predicted in their reports",
    id_token=id_token,
)

I'll help you compare Amazon's recent stock performance to analyst predictions. Let me gather the relevant information by looking at Amazon's recent financial reports and current stock data.Let me get some historical stock data to better understand the recent performance trends.Based on my analysis of Amazon's recent stock performance and the available analyst predictions and guidance, here's a comprehensive comparison:

## Current Stock Performance vs. Analyst Predictions

### **Current Stock Position (April 8, 2026)**
- **Current Price**: $220.86
- **Previous Close**: $213.77 (+3.3% daily gain)
- **Day Range**: $219.62 - $226.00

### **Recent Performance Trends**

**Strong Performance in 2024:**
- Amazon's stock showed remarkable growth in 2024, rising from around $150 at the start to over $230 by year-end
- The stock peaked at around $250+ in late 2024/early 2025
- This represents approximately **47% growth** through 2024, significantly outperforming the broader market

**2025-2026 

## Cell 7: Query 4 — Multi-Source Research

> *"I'm researching AMZN -- give me the current price and any relevant information about their AI business"*

Expected: Agent calls realtime tool + RAG tool, combines price with AI business insights.

In [7]:
response_4 = invoke_agent(
    "I'm researching AMZN -- give me the current price"
    " and any relevant information about their AI business",
    id_token=id_token,
)

I'll help you research Amazon (AMZN) by getting the current stock price and finding information about their AI business.## Amazon (AMZN) Current Stock Information

**Current Price:** $220.89 USD
- **Previous Close:** $213.77 (+3.33% gain)
- **Day Range:** $219.62 - $226.00
- **Volume:** 18.16M shares

## Amazon's AI Business - Key Highlights

Amazon is making massive investments in AI across multiple fronts, with over **1,000 GenAI applications** being built across the company. Here are the key areas:

### **AWS AI Infrastructure & Services**
- **Trainium2 chips**: Custom AI chips with 150% quarter-over-quarter growth, now a multi-billion-dollar business that's fully subscribed
- **Project Rainier**: Massive AI compute cluster with nearly 500,000 Trainium2 chips for Anthropic's Claude models
- **Amazon Bedrock**: Expanded with new foundation models including Anthropic's Claude 4, OpenAI models, and DeepSeek-V3.1
- **EC2 P6e-GB200 UltraServers**: New instances using NVIDIA Grace Blackwe

## Cell 8: Query 5 — Document-Only Query

> *"What is the total amount of office space Amazon owned in North America in 2024?"*

Expected: Agent calls RAG tool only, retrieves data from Amazon 2024 Annual Report.

In [8]:
response_5 = invoke_agent(
    "What is the total amount of office space Amazon owned in North America in 2024?",
    id_token=id_token,
)

I'll search Amazon's financial documents to find information about their office space ownership in North America for 2024.Based on Amazon's 2024 Annual Report, the total amount of office space Amazon owned in North America in 2024 was **9,104 thousand square feet** (or 9.104 million square feet).

This information comes from the Properties section of their annual report, which shows their facilities breakdown as of December 31, 2024. The report indicates that Amazon owned 9,104 thousand square feet of office space in North America, while also leasing an additional 29,551 thousand square feet of office space in the region.


## Cell 9: Multi-Turn Demo

Two messages in the same `thread_id` to demonstrate conversation memory
via `AgentCoreMemorySaver`.

In [9]:
thread_id = str(uuid.uuid4())
print(f"Thread ID: {thread_id}")
print("=" * 60)

print("\n--- Turn 1 ---")
print("User: What is the current stock price of Amazon?\n")
turn_1 = invoke_agent(
    "What is the current stock price of Amazon?",
    id_token=id_token,
    thread_id=thread_id,
)

print("\n--- Turn 2 (follow-up in same thread) ---")
print("User: How does that compare to its price 6 months ago?\n")
turn_2 = invoke_agent(
    "How does that compare to its price 6 months ago?",
    id_token=id_token,
    thread_id=thread_id,
)

Thread ID: 253901f7-fe01-4726-a5fa-0887016ecd0d

--- Turn 1 ---
User: What is the current stock price of Amazon?

I'll get the current stock price for Amazon (AMZN) for you.Amazon's current stock price is **$220.64 USD**.

Here are some additional details:
- Previous close: $213.77
- Day's high: $226.00
- Day's low: $219.62
- Trading volume: 18,199,746 shares
- The stock is up about $6.87 (+3.2%) from the previous close

The data is current as of April 8, 2026, 3:18 PM UTC.

--- Turn 2 (follow-up in same thread) ---
User: How does that compare to its price 6 months ago?

I'll get Amazon's historical stock price from 6 months ago to compare with the current price.Let me try getting a broader date range around 6 months ago to ensure we capture the data:Great! I found the historical data. Looking at Amazon's stock price from 6 months ago (around October 8, 2025), here's the comparison:

**Price Comparison:**
- **Current price (April 8, 2026):** $220.64
- **6 months ago (October 8, 2025):*

## Cell 10: Langfuse Traces

Langfuse captures full traces of every agent invocation — LLM calls, tool
invocations, and graph transitions.

### Langfuse UI Screenshots

**Trace 1: Real-time stock price query** — Full LangGraph span tree with `retrieve_realtime_stock_price` tool call, ChatBedrock LLM invocations, latency (5.89s), cost ($0.008), and graph visualization.

![Langfuse trace — real-time stock price](images/langfuse-trace-realtime.png)

**Trace 2: Multi-turn conversation** — Follow-up question with multiple agent iterations, two `retrieve_historical_stock_price` tool calls, and richer token usage (17.91s, $0.021).

![Langfuse trace — multi-turn conversation](images/langfuse-trace-multiturn.png)

### API Traces

In [10]:
import os

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    from langfuse import Langfuse

    lf = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
    )

    traces = lf.api.trace.list(page=1, limit=5)
    print(f"Recent Langfuse traces ({len(traces.data)} found):")
    print("-" * 80)
    for t in traces.data:
        print(f"  ID: {t.id}")
        print(f"  Name: {t.name}")
        print(f"  Timestamp: {t.timestamp}")
        print(f"  URL: {LANGFUSE_HOST}/trace/{t.id}")
        print("-" * 80)
else:
    print("Langfuse keys not set — skipping trace retrieval.")
    print("Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to fetch traces.")
    print("\nSee screenshots in notebooks/images/ for trace examples.")

Recent Langfuse traces (5 found):
--------------------------------------------------------------------------------
  ID: 1c2ff39c0c189ea98e855901d0275a3d
  Name: LangGraph
  Timestamp: 2026-04-08 15:18:15.987000+00:00
  URL: https://cloud.langfuse.com/trace/1c2ff39c0c189ea98e855901d0275a3d
--------------------------------------------------------------------------------
  ID: 25d0161d4196651d6835d4b2fe9d8fa8
  Name: LangGraph
  Timestamp: 2026-04-08 15:17:59.764000+00:00
  URL: https://cloud.langfuse.com/trace/25d0161d4196651d6835d4b2fe9d8fa8
--------------------------------------------------------------------------------
  ID: d9c314b7ae5c28ff5c34c9ade90c81fc
  Name: LangGraph
  Timestamp: 2026-04-08 15:17:42.930000+00:00
  URL: https://cloud.langfuse.com/trace/d9c314b7ae5c28ff5c34c9ade90c81fc
--------------------------------------------------------------------------------
  ID: 7c7721e4c9ce9897dd8986c62d9b5563
  Name: LangGraph
  Timestamp: 2026-04-08 15:17:18.806000+00:00
  URL: http